In [5]:
%%duckdb


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,455,1 Ave & E 44 St,40.750020,-73.969053,265,Stanton St & Chrystie St,40.722293,-73.991475,18660,Subscriber,1960.0,2,2015-01-01 00:01:00,2015-01-01 00:24:00,23
1,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
2,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6


## Problem 1: Trip duration

### Part 1: Build a Regression Model

Build a regression to predict trip duration by using
- Day of time
- Distance between start and end stations (there might be more than one way to measure it)
- Hour of day
- Weekend indicator
- Don't forget to model bias (this one is intentionally not used in lecture)
- Also any thing you want to end

### Part 2: Experiment Design

- Ensure that you properly design your experiment to report unbiased performance metric you choose

### Part 3 [Optional]: Visualize

- Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
- Estimate trip duration for those say 10 trips
- Visualize them on map using `pydeck` by using redish color for slower trips and greener for faster trips.

In [1]:
%%duckdb -o trip_data

with dataset as (
    select 
        * exclude(tripduration, starttime, stoptime),
        strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as start_at,
        strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as stop_at
    from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
    using sample 100000 rows
),
features as (
    select
        date_diff('second', start_at, stop_at) / 60.0 as duration_min,
        hour(start_at) as hour_of_day,
        isodow(start_at) as day_of_week,
        case when isodow(start_at) in (6, 7) then 1 else 0 end as is_weekend,

        6371 * 2 * asin(
            sqrt(
                pow(sin(radians("end station latitude" - "start station latitude") / 2), 2) +
                cos(radians("start station latitude")) *
                cos(radians("end station latitude")) *
                pow(sin(radians("end station longitude" - "start station longitude") / 2), 2)
            )
        ) as distance_km

    from dataset
)
select *
from features
where duration_min between 1 and 180
  and distance_km > 0
  and distance_km < 30

,duration_min,hour_of_day,day_of_week,is_weekend,distance_km
0,4.783333,8,2,0,0.701966
1,3.783333,14,4,0,0.459564
2,16.000000,15,4,0,1.949361
3,17.733333,10,2,0,3.256755
4,18.933333,8,1,0,3.181286
...,...,...,...,...,...
97617,7.350000,19,1,0,0.828531
97618,14.666667,13,1,0,1.636111
97619,5.883333,21,6,1,0.667438
97620,18.950000,18,5,0,2.848149


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X = trip_data[["day_of_week", "distance_km", "hour_of_day", "is_weekend"]]
y = trip_data["duration_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Bias / intercept:", model.intercept_)
print("Coefficients:", model.coef_)

MAE: 4.588823398145184
RMSE: 9.158754431030594
R2: 0.3769386712867363
Bias / intercept: 2.5237366904838243
Coefficients: [0.02102856 5.24081871 0.06548315 2.36150281]


Bu kısımda modeli daha doğru değerlendirmek için veriyi ikiye ayırdım. Bir kısmını modeli eğitmek için, diğer kısmını ise modeli test etmek için kullandım. Yani model test verisini eğitim sırasında görmedi.

Sonuçları değerlendirirken MAE, RMSE ve R2 değerlerine baktım. MAE, tahminlerin ortalama kaç dakika hata yaptığını gösterdiği için daha anlaşılır bir metrik. RMSE ise büyük hataları daha fazla önemser. R2 değeri de modelin yolculuk süresindeki değişimi ne kadar açıklayabildiğini gösterir.

Bu şekilde modeli aynı veri üzerinde test etmemiş oldum. Bu yüzden sonuçlar modelin hiç görmediği veride nasıl davrandığını daha gerçekçi şekilde gösteriyor.


### Problem 2: Extending Naive Bayesian

### Part 1: Expand the NB Regression Idea to continous variable

$$
P(gender = a, speed_{bike} = x) = P(gender = a) P(speed_{bike} = x | gender = a)
$$

- Note that $P(speed_{bike} = x | gender = a)$ is  continous distribution.
- Expand the idea
- Build a predictive model for estimation biker gender using the bike speed ?

### Part 2: Use Visualization to decide best distribution 

- How should be $P(speed_{bike} = x | gender = a)$ modeled

In [1]:
%%duckdb -o gender_1_duration


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
) where gender =1 
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
1,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6
2,384,Fulton St & Waverly Ave,40.683178,-73.965964,399,Lafayette Ave & St James Pl,40.688515,-73.964763,19610,Subscriber,1969.0,1,2015-01-01 00:04:00,2015-01-01 00:07:00,3


In [3]:
%%duckdb -o gender_speed_data

with dataset as (
    select 
        * exclude(tripduration, starttime, stoptime),
        strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as start_at,
        strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) as stop_at
    from 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
    using sample 100000 rows
),
features as (
    select
        gender,

        date_diff('second', start_at, stop_at) / 60.0 as duration_min,

        6371 * 2 * asin(
            sqrt(
                pow(sin(radians("end station latitude" - "start station latitude") / 2), 2) +
                cos(radians("start station latitude")) *
                cos(radians("end station latitude")) *
                pow(sin(radians("end station longitude" - "start station longitude") / 2), 2)
            )
        ) as distance_km

    from dataset
),
speed_table as (
    select
        gender,
        duration_min,
        distance_km,
        distance_km / (duration_min / 60.0) as speed_kmh
    from features
)
select *
from speed_table
where gender in (1, 2)
  and duration_min between 1 and 180
  and distance_km > 0
  and distance_km < 30
  and speed_kmh between 1 and 40

,gender,duration_min,distance_km,speed_kmh
0,2,23.033333,2.479303,6.458387
1,1,10.516667,1.262970,7.205534
2,2,4.800000,0.730056,9.125694
3,1,8.850000,1.323340,8.971795
4,1,15.683333,2.419250,9.255368
...,...,...,...,...
86097,1,4.283333,0.784890,10.994565
86098,1,26.200000,0.942307,2.157954
86099,1,16.900000,2.306700,8.189468
86100,1,9.483333,1.251287,7.916753


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = gender_speed_data[["speed_kmh"]]
y = gender_speed_data["gender"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

gender_pred = nb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, gender_pred))
print(confusion_matrix(y_test, gender_pred))
print(classification_report(y_test, gender_pred))

Accuracy: 0.7576215086231927
[[13047     0]
 [ 4174     0]]
              precision    recall  f1-score   support

           1       0.76      1.00      0.86     13047
           2       0.00      0.00      0.00      4174

    accuracy                           0.76     17221
   macro avg       0.38      0.50      0.43     17221
weighted avg       0.57      0.76      0.65     17221



/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Bu problemde speed yani hız değeri sürekli bir değişken. Bu yüzden her hız değeri için ayrı ayrı olasılık tablosu oluşturmak çok mantıklı olmaz. Çünkü hız 10, 10.5, 11.2 gibi çok farklı değerler alabilir.

Bu yüzden Gaussian Naive Bayes kullandım. Bu model, her gender sınıfı için hız değerlerinin normal dağılıma benzer şekilde dağıldığını varsayar.

Model temel olarak verilen hız değerine göre hangi gender sınıfının olasılığı daha yüksekse onu tahmin eder. Yani P(gender | speed) mantığıyla çalışır.
